In [0]:
%run "./2.Transformation"

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pyspark.sql import functions as F

# Consistent visual style — used across all charts
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Threshold for opex per mwh to be considered uneconomic (More research is needed for this to be an accurate threshold)
# A Market Average for Wholesale mgwh rate in the US
UNECONOMIC_OPEX_THRESHOLD = 50

In [0]:
# Loading Tables from Transformation
plant_economics = spark.table("plant_economics_final")
utility_summary = spark.table("utility_summary_final")

print("plant_economics rows:", plant_economics.count())
print("utility_summary rows:", utility_summary.count())

## Economically not viable plants

In [0]:
top_uneconomic = spark.sql(f"""
    SELECT
        plant_name_eia AS plant_name,
        utility_name_eia AS utility,
        state,
        ferc_primary_fuel  AS fuel,
        ROUND(opex_per_mwh_safe, 0) AS cost_per_mwh_usd,
        ROUND(opex_per_mwh_safe / {UNECONOMIC_OPEX_THRESHOLD}, 1) AS times_over_threshold,
        CASE
            WHEN zombie_plant = TRUE THEN 'Zombie — paying with no output'
            WHEN opex_per_mwh_safe > 100  THEN 'Very high cost — over $100/MWh'
            ELSE 'Above the $50/MWh threshold'
        END AS why_is_it_uneconomic
    FROM plant_economics_final
    WHERE report_year = (SELECT MAX(report_year) FROM plant_economics_final)
      AND opex_per_mwh_safe IS NOT NULL
    ORDER BY opex_per_mwh_safe DESC
    LIMIT 20
""")

display(top_uneconomic)

### Which fossil fuel is consistently the most expensive 

In [0]:
df = spark.sql("""
    SELECT
        ferc_primary_fuel,
        ROUND(PERCENTILE(opex_per_mwh_safe, 0.5), 1) AS median_opex
    FROM plant_economics_final
    WHERE opex_per_mwh_safe IS NOT NULL
      AND opex_per_mwh_safe BETWEEN 0 AND 200
      AND ferc_primary_fuel IN ('coal', 'gas', 'oil')
      AND report_year >= 2018
    GROUP BY ferc_primary_fuel
    ORDER BY median_opex DESC
""").toPandas()

print("Median opex per MWh by fuel (2018+)")
display(df)

### Cost trend by fuel over time

In [0]:
# oil
cost_trend = spark.sql(f"""
    SELECT
        report_year AS year,
        ferc_primary_fuel AS fuel,
        COUNT(*) AS plant_count,
        ROUND(PERCENTILE(opex_per_mwh_safe, 0.5), 0) AS median_opex_per_mwh,
        SUM(CASE WHEN opex_per_mwh_safe > {UNECONOMIC_OPEX_THRESHOLD} THEN 1 ELSE 0 END) AS uneconomic_plant_count
    FROM plant_economics_final
    WHERE opex_per_mwh_safe IS NOT NULL
      AND opex_per_mwh_safe BETWEEN 0 AND 200
      AND ferc_primary_fuel = 'oil'
    GROUP BY report_year, ferc_primary_fuel
    ORDER BY report_year, ferc_primary_fuel
""")

cost_trend_pd = cost_trend.toPandas()

fig, ax2 = plt.subplots()
color = 'tab:red'
ax2.set_ylabel('Uneconomic Plant Count', color=color)
ax2.bar(cost_trend_pd['year'], cost_trend_pd['uneconomic_plant_count'], alpha=0.3, color=color, label='Uneconomic Plant Count')
ax2.tick_params(axis='y', labelcolor=color)

fig.tight_layout()
plt.show()

In [0]:
# coal
cost_trend = spark.sql(f"""
    SELECT
        report_year AS year,
        ferc_primary_fuel AS fuel,
        COUNT(*) AS plant_count,
        ROUND(PERCENTILE(opex_per_mwh_safe, 0.5), 0) AS median_opex_per_mwh,
        SUM(CASE WHEN opex_per_mwh_safe > {UNECONOMIC_OPEX_THRESHOLD} THEN 1 ELSE 0 END) AS uneconomic_plant_count
    FROM plant_economics_final
    WHERE opex_per_mwh_safe IS NOT NULL
      AND opex_per_mwh_safe BETWEEN 0 AND 200
      AND ferc_primary_fuel = 'coal'
    GROUP BY report_year, ferc_primary_fuel
    ORDER BY report_year, ferc_primary_fuel
""")

cost_trend_pd = cost_trend.toPandas()

fig, ax2 = plt.subplots()
color = 'tab:red'
ax2.set_ylabel('Uneconomic Plant Count', color=color)
ax2.bar(cost_trend_pd['year'], cost_trend_pd['uneconomic_plant_count'], alpha=0.3, color=color, label='Uneconomic Plant Count')
ax2.tick_params(axis='y', labelcolor=color)

fig.tight_layout()
plt.show()

In [0]:
# gas
cost_trend = spark.sql(f"""
    SELECT
        report_year AS year,
        ferc_primary_fuel AS fuel,
        COUNT(*) AS plant_count,
        ROUND(PERCENTILE(opex_per_mwh_safe, 0.5), 0) AS median_opex_per_mwh,
        SUM(CASE WHEN opex_per_mwh_safe > {UNECONOMIC_OPEX_THRESHOLD} THEN 1 ELSE 0 END) AS uneconomic_plant_count
    FROM plant_economics_final
    WHERE opex_per_mwh_safe IS NOT NULL
      AND opex_per_mwh_safe BETWEEN 0 AND 200
      AND ferc_primary_fuel = 'gas'
    GROUP BY report_year, ferc_primary_fuel
    ORDER BY report_year, ferc_primary_fuel
""")

cost_trend_pd = cost_trend.toPandas()

fig, ax2 = plt.subplots()
color = 'tab:red'
ax2.set_ylabel('Uneconomic Plant Count', color=color)
ax2.bar(cost_trend_pd['year'], cost_trend_pd['uneconomic_plant_count'], alpha=0.3, color=color, label='Uneconomic Plant Count')
ax2.tick_params(axis='y', labelcolor=color)

fig.tight_layout()
plt.show()

### Where does the money go?


In [0]:
uneconomic_by_class = spark.sql(f"""
    SELECT
        utility_class,
        COUNT(*)                                                AS uneconomic_plant_years,
        ROUND(AVG(opex_per_mwh_safe), 0)                        AS avg_opex_per_mwh,
        ROUND(SUM(opex_production_total) / 1e9, 2)              AS total_opex_billion_usd
    FROM plant_economics_final
    WHERE opex_per_mwh_safe > {UNECONOMIC_OPEX_THRESHOLD}
      AND report_year >= 2020
    GROUP BY utility_class
    ORDER BY uneconomic_plant_years DESC
""")

display(uneconomic_by_class)

### Zombie plants by utility class 

In [0]:
zombies_by_class = spark.sql("""
    SELECT
        utility_class,
        COUNT(*)                                                AS zombie_plant_years,
        COUNT(DISTINCT plant_id_pudl)                           AS distinct_zombie_plants,
        ROUND(SUM(opex_production_total) / 1e6, 1)              AS total_opex_million_usd
    FROM plant_economics_final
    WHERE zombie_plant = TRUE
    GROUP BY utility_class
    ORDER BY zombie_plant_years DESC
""")

display(zombies_by_class)

This confirms that almost all of zombie plants are partly government owned

### Median heat Rate by fuel Type

In [0]:
heat_rate_summary = spark.sql("""
    SELECT
        ferc_primary_fuel AS fuel,
        COUNT(*) AS plant_count,
        ROUND(PERCENTILE(plant_heat_rate, 0.5), 1) AS median_heat_rate
    FROM plant_economics_final
    WHERE plant_heat_rate BETWEEN 5 AND 20
      AND report_year >= 2020
    GROUP BY ferc_primary_fuel
    ORDER BY median_heat_rate DESC
""")

display(heat_rate_summary)

This insights shows that coal based fossil fuel plants creates most heat beacuse of just the sheer amount of plants

In [0]:
display(
    spark.sql("""
        SELECT
            utility_name_eia AS utility,
            utility_class AS sector,
            COUNT(*) AS zombie_plant_count
        FROM plant_economics_final
        WHERE zombie_plant = TRUE
          AND report_year = 2025
        GROUP BY utility_name_eia, utility_class
        ORDER BY zombie_plant_count DESC
        LIMIT 10
    """)
)

In 2025 top 10 Utilities that own zombie power plants are all partly government owned

In [0]:
max_opex_by_sector_utility = spark.sql(f"""
    WITH ranked_utilities AS (
        SELECT
            utility_name_eia AS utility,
            utility_class,
            sector_name_eia,
            ROUND(total_opex / total_generation_mwh, 2) AS opex_per_mwh,
            ROW_NUMBER() OVER (PARTITION BY sector_name_eia ORDER BY total_opex / total_generation_mwh DESC) AS rn
        FROM utility_summary_final
        WHERE report_year = {2025}
          AND total_generation_mwh > 0
    )
    SELECT utility, utility_class, sector_name_eia, opex_per_mwh
    FROM ranked_utilities
    WHERE rn = 1
    ORDER BY opex_per_mwh DESC
""")
display(max_opex_by_sector_utility)

In [0]:

# Utilities with highest opex per MWh (min generation threshold)
MIN_GENERATION_MWH = 1000

high_opex_utilities = spark.sql(f"""
    SELECT
        utility_name_eia AS utility,
        utility_class,
        sector_name_eia,
        ROUND(SUM(total_opex) / SUM(total_generation_mwh), 2) AS opex_per_mwh
    FROM utility_summary_final
    WHERE report_year = {2025}
      AND total_generation_mwh > {MIN_GENERATION_MWH}
    GROUP BY utility_name_eia, utility_class
    ORDER BY opex_per_mwh DESC
    LIMIT 10
""")
display(high_opex_utilities)

These are the  most opex heavy utilites sector The Electricity Utilites provide electricity to industrial, commercial and  Residential areas, They are critical infrastructure because without them a lot of other sectors would suffer greatly, that is why their high opex is generally accepted by government, although deeper research can allow us to look which of these plants are actually viable and which are just burning money
